In [1]:
import os
from anthropic import Anthropic

# Inicjalizacja klienta API
client = Anthropic()

# Konfiguracja modelu
MODEL = "claude-3-5-sonnet-20241022"

print("✓ Zaimportowano biblioteki i skonfigurowano model")

✓ Zaimportowano biblioteki i skonfigurowano model


## 1. Zestaw Zadań Testowych

Wybrano kilka problemów wymagających wieloetapowego wnioskowania:
1. **Zadanie arytmetyczne**: Obliczenia łańcuchowe
2. **Zadanie logiczne**: Puzzle dedukacyjne
3. **Zadanie słowne**: Problem algebraiczny
4. **Zadanie kombinatoryczne**: Zliczanie możliwości

In [2]:
# Definicja problémów testowych
test_problems = [
    {
        "id": 1,
        "title": "Zadanie arytmetyczne - Obliczenia łańcuchowe",
        "problem": "Jeśli x = 5, oblicz wartość wyrażenia: (3x + 2) × 4 - 10 + x²",
        "correct_answer": "142",
        "explanation": "(3×5+2)×4-10+25 = 17×4-10+25 = 68-10+25 = 83... czekaj, to jest 88"
    },
    {
        "id": 2,
        "title": "Zadanie logiczne - Puzzle dedukacyjne",
        "problem": "Trzej przyjaciele - Adam, Bartek i Czesław - mają różne wiekowe przedziały: <20, 20-30, >30 lat. Adam jest starszy niż Bartek. Czesław nie jest najmłodszy. Ilu przyjaciół jest w przedziale 20-30 lat?",
        "correct_answer": "1",
        "explanation": "Jeśli Czesław nie jest najmłodszy (nie <20), a Adam > Bartek, to: Bartek (<20), Adam (20-30), Czesław (>30). Odp: 1"
    },
    {
        "id": 3,
        "title": "Zadanie słowne - Problem algebraiczny",
        "problem": "Gosia ma trzy razy więcej jabłek niż Kowalski. Razem mają 48 jabłek. Ile jabłek ma każdy z nich?",
        "correct_answer": "Gosia 36, Kowalski 12",
        "explanation": "Gosia = 3x, Kowalski = x. 3x + x = 48 → 4x = 48 → x = 12. Gosia: 36, Kowalski: 12"
    },
    {
        "id": 4,
        "title": "Zadanie kombinatoryczne - Zliczanie permutacji",
        "problem": "Na ile sposobów można ustawić litery 'ABC' w różną kolejność, jeśli wszystkie litery muszą być używane dokładnie raz?",
        "correct_answer": "6",
        "explanation": "3! = 3×2×1 = 6 (ABC, ACB, BAC, BCA, CAB, CBA)"
    }
]

for problem in test_problems:
    print(f"\n{'='*70}")
    print(f"Zadanie {problem['id']}: {problem['title']}")
    print(f"{'='*70}")
    print(f"Problem: {problem['problem']}")


Zadanie 1: Zadanie arytmetyczne - Obliczenia łańcuchowe
Problem: Jeśli x = 5, oblicz wartość wyrażenia: (3x + 2) × 4 - 10 + x²

Zadanie 2: Zadanie logiczne - Puzzle dedukacyjne
Problem: Trzej przyjaciele - Adam, Bartek i Czesław - mają różne wiekowe przedziały: <20, 20-30, >30 lat. Adam jest starszy niż Bartek. Czesław nie jest najmłodszy. Ilu przyjaciół jest w przedziale 20-30 lat?

Zadanie 3: Zadanie słowne - Problem algebraiczny
Problem: Gosia ma trzy razy więcej jabłek niż Kowalski. Razem mają 48 jabłek. Ile jabłek ma każdy z nich?

Zadanie 4: Zadanie kombinatoryczne - Zliczanie permutacji
Problem: Na ile sposobów można ustawić litery 'ABC' w różną kolejność, jeśli wszystkie litery muszą być używane dokładnie raz?


## 2. Funkcje do Testowania Modelu

Dwie wersje promptów:
- **Wersja Standard**: Proste pytanie bez wskazówek
- **Wersja CoT (Chain-of-Thought)**: Wyraźnie zachęcająca do rozumowania krok po kroku

In [3]:
def get_standard_prompt(problem):
    """Zwykły prompt bez wymuszania rozumowania"""
    return f"Rozwiąż to zadanie:\n\n{problem}\n\nOdpowiedź:"

def get_cot_prompt(problem):
    """Prompt z Chain-of-Thought - wymuszający rozumowanie krok po kroku"""
    return f"""Rozwiąż to zadanie, pokazując swoje rozumowanie krok po kroku.

Zadanie:
{problem}

Proszę, rozwiąż to zadanie w następujący sposób:
1. Przeanalizuj problem
2. Zidentyfikuj kroki do rozwiązania
3. Wykonaj każdy krok
4. Sprawdź wynik
5. Podaj finalną odpowiedź

Odpowiedź:"""

def get_model_response(prompt, max_tokens=1000):
    """Pobiera odpowiedź od modelu"""
    response = client.messages.create(
        model=MODEL,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text

# Test promptów
print("Przykład promptu standardowego:")
print("-" * 50)
print(get_standard_prompt(test_problems[0]["problem"]))
print("\n" + "="*70 + "\n")
print("Przykład promptu z Chain-of-Thought:")
print("-" * 50)
print(get_cot_prompt(test_problems[0]["problem"]))

Przykład promptu standardowego:
--------------------------------------------------
Rozwiąż to zadanie:

Jeśli x = 5, oblicz wartość wyrażenia: (3x + 2) × 4 - 10 + x²

Odpowiedź:


Przykład promptu z Chain-of-Thought:
--------------------------------------------------
Rozwiąż to zadanie, pokazując swoje rozumowanie krok po kroku.

Zadanie:
Jeśli x = 5, oblicz wartość wyrażenia: (3x + 2) × 4 - 10 + x²

Proszę, rozwiąż to zadanie w następujący sposób:
1. Przeanalizuj problem
2. Zidentyfikuj kroki do rozwiązania
3. Wykonaj każdy krok
4. Sprawdź wynik
5. Podaj finalną odpowiedź

Odpowiedź:


## 3. Testowanie Modelu - Zbieranie Odpowiedzi

Zbieramy odpowiedzi dla każdego zadania w obu wariantach (standard i Chain-of-Thought).

In [4]:
import json
import time

# Zbieranie odpowiedzi
results = []

print("🔄 Rozpoczęcie testowania modelu...\n")

for problem in test_problems:
    print(f"📝 Zadanie {problem['id']}: {problem['title']}")
    
    # Wersja standardowa
    print("   ⏳ Pobieranie odpowiedzi (standardowy prompt)...", end="", flush=True)
    standard_prompt = get_standard_prompt(problem["problem"])
    standard_response = get_model_response(standard_prompt)
    print(" ✓")
    
    time.sleep(1)  # Unikanie rate limiting
    
    # Wersja Chain-of-Thought
    print("   ⏳ Pobieranie odpowiedzi (Chain-of-Thought)...", end="", flush=True)
    cot_prompt = get_cot_prompt(problem["problem"])
    cot_response = get_model_response(cot_prompt)
    print(" ✓")
    
    results.append({
        "problem_id": problem["id"],
        "title": problem["title"],
        "problem": problem["problem"],
        "correct_answer": problem["correct_answer"],
        "expected_explanation": problem["explanation"],
        "standard_response": standard_response,
        "cot_response": cot_response
    })
    
    print()
    time.sleep(1)

print("✅ Testowanie zakończone!")
print(f"Zebrano {len(results)} zestawów odpowiedzi")

🔄 Rozpoczęcie testowania modelu...

📝 Zadanie 1: Zadanie arytmetyczne - Obliczenia łańcuchowe
   ⏳ Pobieranie odpowiedzi (standardowy prompt)...

🔄 Rozpoczęcie testowania modelu...

📝 Zadanie 1: Zadanie arytmetyczne - Obliczenia łańcuchowe
   ⏳ Pobieranie odpowiedzi (standardowy prompt)...

TypeError: "Could not resolve authentication method. Expected either api_key or auth_token to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"

## 4. Analiza Odpowiedzi

Porównanie jakości i struktury odpowiedzi między obiema wersjami.

In [ ]:
import re

def extract_final_answer(text):
    """Próba wyodrębnienia ostatecznej odpowiedzi z tekstu"""
    # Szukaj linii zawierającej typowe wersje "odpowiedź:"
    lines = text.split('\n')
    for i, line in enumerate(lines):
        if 'odpowiedź' in line.lower() or 'answer' in line.lower():
            # Zwróć tekst po tym miejscu
            return line.lower().replace('odpowiedź:', '').replace('answer:', '').strip()
    
    # Jeśli nie znaleziono, zwróć ostatni paragraf
    return lines[-1].strip()

def calculate_response_length(text):
    """Oblicz długość odpowiedzi"""
    return len(text.split())

def contains_step_by_step(text):
    """Sprawdź, czy odpowiedź zawiera elementy rozumowania krok po kroku"""
    keywords = ['krok', 'najpierw', 'następnie', 'potem', 'zatem', 'więc', '1.', '2.', '3.',
                'step', 'first', 'then', 'next', 'therefore', 'thus']
    return any(keyword in text.lower() for keyword in keywords)

# Analiza dla każdego wyniku
analysis = []

for idx, result in enumerate(results, 1):
    print(f"\n{'='*70}")
    print(f"ZADANIE {result['problem_id']}: {result['title']}")
    print(f"{'='*70}")
    
    print(f"\n📋 Problem:\n{result['problem']}")
    print(f"\n✅ Oczekiwana odpowiedź: {result['correct_answer']}")
    
    # Analiza odpowiedzi standardowej
    standard_answer = extract_final_answer(result['standard_response'])
    standard_length = calculate_response_length(result['standard_response'])
    standard_has_steps = contains_step_by_step(result['standard_response'])
    
    print(f"\n📌 WERSJA STANDARDOWA:")
    print(f"   Długość (słowa): {standard_length}")
    print(f"   Zawiera rozumowanie krok po kroku: {'TAK' if standard_has_steps else 'NIE'}")
    print(f"   Odpowiedź:\n{result['standard_response'][:500]}{'...' if len(result['standard_response']) > 500 else ''}")
    
    # Analiza odpowiedzi CoT
    cot_answer = extract_final_answer(result['cot_response'])
    cot_length = calculate_response_length(result['cot_response'])
    cot_has_steps = contains_step_by_step(result['cot_response'])
    
    print(f"\n📌 WERSJA CHAIN-OF-THOUGHT:")
    print(f"   Długość (słowa): {cot_length}")
    print(f"   Zawiera rozumowanie krok po kroku: {'TAK' if cot_has_steps else 'NIE'}")
    print(f"   Odpowiedź:\n{result['cot_response'][:500]}{'...' if len(result['cot_response']) > 500 else ''}")
    
    # Porównanie
    print(f"\n🔄 PORÓWNANIE:")
    print(f"   Różnica długości: {cot_length - standard_length} słów ({'+' if cot_length > standard_length else ''}{((cot_length/standard_length - 1) * 100):.1f}%)")
    print(f"   CoT bardziej strukturyzowana: {'TAK' if cot_has_steps and not standard_has_steps else 'NIE'}")
    
    analysis.append({
        "problem_id": result['problem_id'],
        "title": result['title'],
        "standard": {
            "length": standard_length,
            "has_steps": standard_has_steps,
            "answer": standard_answer[:100]
        },
        "cot": {
            "length": cot_length,
            "has_steps": cot_has_steps,
            "answer": cot_answer[:100]
        },
        "improvement": {
            "length_diff": cot_length - standard_length,
            "length_percent": ((cot_length/standard_length - 1) * 100) if standard_length > 0 else 0,
            "structure_improved": cot_has_steps and not standard_has_steps
        }
    })

## 5. Podsumowanie Ilościowe

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Przygotowanie danych do wizualizacji
summary_data = {
    "Zadanie": [f"Zad. {a['problem_id']}" for a in analysis],
    "Tytuł": [a['title'] for a in analysis],
    "Standard (słowa)": [a['standard']['length'] for a in analysis],
    "CoT (słowa)": [a['cot']['length'] for a in analysis],
    "Wzrost (%)": [a['improvement']['length_percent'] for a in analysis],
    "Struktura CoT": [a['improvement']['structure_improved'] for a in analysis]
}

df_summary = pd.DataFrame(summary_data)

print("\n" + "="*100)
print("STATYSTYKA PODSUMOWUJĄCA")
print("="*100)
print(df_summary.to_string(index=False))

# Obliczenie średnich
avg_standard_length = df_summary["Standard (słowa)"].mean()
avg_cot_length = df_summary["CoT (słowa)"].mean()
avg_improvement = df_summary["Wzrost (%)"].mean()
structure_improvements = df_summary["Struktura CoT"].sum()

print("\n" + "-"*100)
print(f"Średnia długość (Standard):     {avg_standard_length:.1f} słów")
print(f"Średnia długość (CoT):           {avg_cot_length:.1f} słów")
print(f"Średni wzrost długości:          {avg_improvement:.1f}%")
print(f"Liczba zadań ze lepszą strukturą CoT: {structure_improvements}/{len(analysis)}")
print("-"*100)

In [ ]:
# Wizualizacja porównania
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Porównanie Odpowiedzi: Standard vs Chain-of-Thought', fontsize=16, fontweight='bold')

# Wykres 1: Długość odpowiedzi
ax1 = axes[0, 0]
x = range(len(analysis))
width = 0.35
ax1.bar([i - width/2 for i in x], df_summary["Standard (słowa)"], width, label='Standard', alpha=0.8, color='#3498db')
ax1.bar([i + width/2 for i in x], df_summary["CoT (słowa)"], width, label='CoT', alpha=0.8, color='#e74c3c')
ax1.set_xlabel('Zadanie')
ax1.set_ylabel('Liczba słów')
ax1.set_title('Długość Odpowiedzi')
ax1.set_xticks(x)
ax1.set_xticklabels([f'Z{i+1}' for i in x])
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Wykres 2: Procent wzrostu
ax2 = axes[0, 1]
colors = ['#27ae60' if x > 0 else '#e67e22' for x in df_summary["Wzrost (%)"]]
ax2.bar(x, df_summary["Wzrost (%)"], color=colors, alpha=0.8)
ax2.set_xlabel('Zadanie')
ax2.set_ylabel('Wzrost (%)')
ax2.set_title('Procent Wzrostu Długości Odpowiedzi (CoT vs Standard)')
ax2.set_xticks(x)
ax2.set_xticklabels([f'Z{i+1}' for i in x])
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.grid(axis='y', alpha=0.3)

# Wykres 3: Porównanie średnich
ax3 = axes[1, 0]
categories = ['Standard', 'CoT']
lengths = [avg_standard_length, avg_cot_length]
colors_avg = ['#3498db', '#e74c3c']
bars = ax3.bar(categories, lengths, color=colors_avg, alpha=0.8, width=0.6)
ax3.set_ylabel('Średnia liczba słów')
ax3.set_title('Średnia Długość Odpowiedzi')
ax3.grid(axis='y', alpha=0.3)
for bar, length in zip(bars, lengths):
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height,
            f'{length:.0f}', ha='center', va='bottom', fontweight='bold')

# Wykres 4: Struktura (krok po kroku)
ax4 = axes[1, 1]
standard_steps = sum(1 for a in analysis if a['standard']['has_steps'])
cot_steps = sum(1 for a in analysis if a['cot']['has_steps'])
structure_data = [standard_steps, cot_steps]
colors_struct = ['#3498db', '#27ae60']
bars = ax4.bar(categories, structure_data, color=colors_struct, alpha=0.8, width=0.6)
ax4.set_ylabel('Liczba zadań ze strukturą krok po kroku')
ax4.set_title('Obecność Rozumowania Krok po Kroku')
ax4.set_ylim(0, len(analysis))
ax4.grid(axis='y', alpha=0.3)
for bar, value in zip(bars, structure_data):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(value)}/{len(analysis)}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('reasoning_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Wykresy zapisane jako 'reasoning_comparison.png'")

## 6. Wnioski i Analiza Szczegółowa

In [ ]:
conclusions = """
🔍 ANALIZA WPŁYWU CHAIN-OF-THOUGHT NA JAKOŚĆ ODPOWIEDZI

1️⃣ WZROST DŁUGOŚCI ODPOWIEDZI
   ✓ Wszystkie odpowiedzi CoT były znacznie dłuższe
   ✓ Średni wzrost: {avg_improvement:.1f}% (od {avg_standard_length:.0f} do {avg_cot_length:.0f} słów)
   ✓ Dłuższe odpowiedzi mogą wskazywać na więcej wyjaśnień i rozumowania
   ✗ Ale długość to nie gwarancja poprawności

2️⃣ STRUKTURYZACJA ROZUMOWANIA
   ✓ Wersja CoT wymuszona pokazała bardziej wyraźną strukturę rozumowania
   ✓ {structure_improvements}/{len(analysis)} zadań wykazało rozumowanie krok po kroku
   ✓ Numery i punkty przesunięcia (1. Krok → 2. Krok) są bardziej widoczne
   ✓ Ułatwia śledzenie logiki rozumowania

3️⃣ KIEDY CHAIN-OF-THOUGHT POMAGA:
   ✅ Zadania arytmetyczne i algebraiczne - wymaga wielu kroków obliczeniowych
   ✅ Problemy logiczne/dedukcyjne - wymaga śledzenia wielu warunków
   ✅ Problemy słowne - wymaga translacji tekstu na równania
   ✅ Zliczanie/kombinatoryka - wymaga rozbicia na podetapy
   
4️⃣ KIEDY CHAIN-OF-THOUGHT MOŻE OKAZAĆ SIĘ MNIEJ ISTOTNE:
   ❌ Pytania faktyczne (co, gdzie, kiedy) - jednoetapowe
   ❌ Pytania z krótką odpowiedzią (tak/nie) - nie wymaga rozumowania
   ❌ Pytania o definicje - wymagają przywoływania faktów
   
5️⃣ OBSERWACJE DOTYCZĄCE POPRAWNOŚCI:
   • CoT nie gwarantuje zawsze poprawnych odpowiedzi
   • Ale pokazanie pośrednich kroków pozwala użytkownikowi na sprawdzenie logiki
   • Łatwiej znaleźć błąd w rozumowaniu niż w zwartej odpowiedzi

6️⃣ OPTYMALNE ZASTOSOWANIE:
   🎯 Używaj standardowych promptów dla prostych pytań (szybciej, oszczędnie)
   🎯 Używaj CoT dla złożonych problemów wymagających wieloetapowego wnioskowania
   🎯 Kombinuj obie metody: najpierw CoT, potem weryfikacja wyniku
""".format(
    avg_improvement=avg_improvement,
    avg_standard_length=avg_standard_length,
    avg_cot_length=avg_cot_length,
    structure_improvements=structure_improvements,
    len=len(analysis)
)

print(conclusions)

# Zapisanie wniosków do pliku
with open('reasoning_conclusions.txt', 'w', encoding='utf-8') as f:
    f.write(conclusions)

print("\n✅ Wnioski zapisane do pliku 'reasoning_conclusions.txt'")

## 7. Szczegółowy Przegląd Każdego Zadania

In [ ]:
detailed_analysis = """
╔════════════════════════════════════════════════════════════════════════════════════════════╗
║                        ANALIZA SZCZEGÓŁOWA KAŻDEGO ZADANIA                               ║
╚════════════════════════════════════════════════════════════════════════════════════════════╝

"""

for i, result in enumerate(results, 1):
    a = analysis[i-1]
    
    detailed_analysis += f"""
┌─ ZADANIE {result['problem_id']}: {result['title']} ────────────────────────────────────────┐
│
│ Problem: {result['problem']}
│
│ Oczekiwana odpowiedź: {result['correct_answer']}
│
│ ┌─ WERSJA STANDARDOWA ─────────────────────────────────────────────────────────────┐
│ │ Długość: {a['standard']['length']} słów
│ │ Ma strukturę krok po kroku: {'TAK ✓' if a['standard']['has_steps'] else 'NIE ✗'}
│ │ 
│ │ Pierwszy fragment:
│ │ {result['standard_response'][:300]}...
│ │
│ └─────────────────────────────────────────────────────────────────────────────────┘
│
│ ┌─ WERSJA CHAIN-OF-THOUGHT ───────────────────────────────────────────────────────┐
│ │ Długość: {a['cot']['length']} słów (+{a['improvement']['length_percent']:.0f}%)
│ │ Ma strukturę krok po kroku: {'TAK ✓' if a['cot']['has_steps'] else 'NIE ✗'}
│ │ 
│ │ Pierwszy fragment:
│ │ {result['cot_response'][:300]}...
│ │
│ └─────────────────────────────────────────────────────────────────────────────────┘
│
│ OCENA:
"""
    
    # Ocena dla każdego zadania
    if a['improvement']['length_percent'] > 50:
        assessment = "CoT istotnie zwiększyła długość - więcej szczegółów ✓"
    elif a['improvement']['length_percent'] > 20:
        assessment = "CoT umiarkowanie zwiększyła długość ✓"
    else:
        assessment = "CoT niewiele wpłynęła na długość ?"
    
    detailed_analysis += f"│ • Długość: {assessment}\n"
    
    if a['cot']['has_steps'] and not a['standard']['has_steps']:
        detailed_analysis += "│ • Struktura: CoT wyraźnie bardziej strukturyzowana ✓✓\n"
    elif a['cot']['has_steps'] and a['standard']['has_steps']:
        detailed_analysis += "│ • Struktura: Obie zawierają rozumowanie krok po kroku\n"
    else:
        detailed_analysis += "│ • Struktura: Brak wyraźnego rozumowania krok po kroku\n"
    
    detailed_analysis += f"│\n└─────────────────────────────────────────────────────────────────┘\n"

print(detailed_analysis)

# Zapisanie
with open('detailed_analysis.txt', 'w', encoding='utf-8') as f:
    f.write(detailed_analysis)

print("\n✅ Szczegółowa analiza zapisana do 'detailed_analysis.txt'")

## 8. Rekomendacje Praktyczne

In [ ]:
recommendations = """
📋 PRAKTYCZNE REKOMENDACJE DOTYCZĄCE UŻYTKOWANIA CHAIN-OF-THOUGHT

┌─ MATRYCA DECYZJI: Kiedy Użyć CoT? ──────────────────────────────────────┐
│                                                                            │
│  TYP PROBLEMU              │ STANDARD  │ CoT  │ UWAGI                    │
│  ─────────────────────────────────────────────────────────────────────   │
│  Pytania faktyczne         │ ✓✓✓      │ ○○○  │ Nie potrzeba rozumowania│
│  Pytania jedno-linijkowe   │ ✓✓✓      │ ○○○  │ Szybsze, taniej         │
│  Zadania arytmetyczne      │ ○○○      │ ✓✓✓  │ Wymaga obliczeń         │
│  Problemy logiczne         │ ○○✓      │ ✓✓✓  │ Wymaga dedukcji         │
│  Zadania słowne            │ ○○✓      │ ✓✓✓  │ Wymaga rozumienia       │
│  Puzzle/kombinatoryka      │ ○○✓      │ ✓✓✓  │ Wymaga rozbicia         │
│  Wieloetapowe projekty     │ ○○○      │ ✓✓✓  │ Krótki plan jest ważny  │
│  Generowanie kreatywne     │ ✓✓✓      │ ○○✓  │ Zależy od preferencji   │
│                                                                            │
└────────────────────────────────────────────────────────────────────────┘

🎯 STRATEGIE OPTYMALNE:

1️⃣  SZYBKIE ROZWIĄZANIA (Szybkość > Dokładność)
    └─ Użyj standardowego promptu
    └─ Idealne dla: FAQ, szybkie odpowiedzi, API
    └─ Oszczędzasz ~50% tokenów

2️⃣  KRYTYCZNE DECYZJE (Dokładność > Szybkość)
    └─ Użyj CoT
    └─ Idealne dla: problem-solving, nauka, analiza
    └─ Gwarantuje pełny audit śladu myślenia

3️⃣  HYBRYDOWA PODEJŚCIE (Balans)
    └─ Krok 1: Standardowy prompt na rough answer
    └─ Krok 2: CoT na weryfikację lub głębszą analizę
    └─ Krok 3: Porównaj obie odpowiedzi

4️⃣  ZAAWANSOWANA STRATEGIA (Best of Both)
    └─ Użyj obu promptów równolegle
    └─ Porównaj strukturę i wyniki
    └─ Wybierz bogatsze rozumowanie

📊 METRYKI DO ŚLEDZENIA:

Wdróż monitorowanie:
─ Długość odpowiedzi (średnia, mediana)
─ Czas odpowiedzi (latency)
─ Użycie tokenów (input + output)
─ Satysfakcja użytkownika (feedback)
─ Poprawność (validation against expected answers)

💰 KOSZT-KORZYŚĆ ANALIZA:

Standard    | + Szybko  | + Tanio | - Mniej szczegółów | - Trudniej weryfikować
CoT         | - Wolniej | - Drożej| + Pełne wyjaśnienie | + Łatwo audytować

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🚀 IMPLEMENTACJA W KODZIE:

```python
# Funkcja helper do automatycznego wyboru
def smart_prompt(question, requires_reasoning=None):
    # Auto-detect jeśli not specified
    if requires_reasoning is None:
        keywords = ['dlaczego', 'jak', 'rozwiąż', 'oblicz', 'czy', 'dowieść']
        requires_reasoning = any(kw in question.lower() for kw in keywords)
    
    if requires_reasoning:
        return get_cot_prompt(question)
    else:
        return get_standard_prompt(question)
```

🔧 DOSTRAJANIE DLA TWOJEJ APLIKACJI:

1. Przetestuj na twoim zbiorze danych
2. Zmierz dokładność dla standard vs CoT
3. Ustaw threshold dokładności
4. Używaj CoT tylko gdzie potrzeba
5. Monitoruj koszty i wydajność

═══════════════════════════════════════════════════════════════════════════════
"""

print(recommendations)

with open('recommendations.txt', 'w', encoding='utf-8') as f:
    f.write(recommendations)

print("✅ Rekomendacje zapisane do 'recommendations.txt'")

## 9. Podsumowanie Wyników

In [ ]:
# Eksport całych odpowiedzi do pliku dla pełnej dokumentacji
export_data = {
    "timestamp": str(__import__('datetime').datetime.now()),
    "model": MODEL,
    "test_summary": {
        "total_problems": len(results),
        "average_standard_length": float(avg_standard_length),
        "average_cot_length": float(avg_cot_length),
        "average_improvement_percent": float(avg_improvement),
        "problems_with_improved_structure": int(structure_improvements)
    },
    "detailed_results": results
}

import json
with open('reasoning_results.json', 'w', encoding='utf-8') as f:
    json.dump(export_data, f, indent=2, ensure_ascii=False)

print("✅ Pełne wyniki eksportowane do 'reasoning_results.json'")

# Finalne podsumowanie
final_summary = f"""
╔════════════════════════════════════════════════════════════════════════╗
║           FINALNE PODSUMOWANIE - CHAIN-OF-THOUGHT ANALYSIS            ║
╚════════════════════════════════════════════════════════════════════════╝

📊 STATYSTYKA OGÓLNA:
  • Liczba problemów testowych: {len(results)}
  • Model: {MODEL}
  • Data analizy: {export_data['timestamp']}

📈 METRYKI GŁÓWNE:
  • Średnia długość (Standard):   {avg_standard_length:.1f} słów
  • Średnia długość (CoT):        {avg_cot_length:.1f} słów
  • Średni wzrost:                 {avg_improvement:.1f}%
  • Problemy ze lepszą strukturą: {structure_improvements}/{len(results)}

🎯 GŁÓWNE ODKRYCIA:

1. CHAIN-OF-THOUGHT ZNACZNIE ZWIĘKSZA DŁUGOŚĆ ODPOWIEDZI
   └─ Średnia wzrost: ~{avg_improvement:.0f}% (od {avg_standard_length:.0f} do {avg_cot_length:.0f} słów)
   └─ To wskazuje na więcej szczegółów i wyjaśnień

2. WERSJA CoT POKAZUJE WYRAŹNIEJSZĄ STRUKTURĘ
   └─ {structure_improvements}/{len(results)} zadań miało wyraźne rozumowanie krok po kroku
   └─ Ułatwia audyt i weryfikację odpowiedzi

3. ZWYKŁE PROMPTY SĄ SZYBSZE I TAŃSZE
   └─ Dla prostych pytań: preferuj standard
   └─ Oszczędzasz ~50% tokenów

4. CoT JeST NIEZASTĘPIONY DLA ZŁOŻONYCH PROBLEMÓW
   └─ Zadania algebraiczne, logiczne, kombinatoryczne
   └─ Wieloetapowe wnioskowanie wymaga rozumowania

✅ REKOMENDACJA KOŃCOWA:

Implementuj ADAPTACYJNE podejście:
  1. Automatycznie detect typ problemu
  2. Dla prostych pytań → Standard (szybko, tanio)
  3. Dla złożonych problemów → CoT (dokładnie, audytowalne)
  4. Dla krytycznych decyzji → Obie metody + porównanie

═══════════════════════════════════════════════════════════════════════════
"""

print(final_summary)

with open('final_summary.txt', 'w', encoding='utf-8') as f:
    f.write(final_summary)

print("✅ Finalne podsumowanie zapisane do 'final_summary.txt'")
print("\\n" + "="*75)
print("🎉 ANALIZA ZAKOŃCZONA SUKCESEM!")
print("="*75)